# BVSIMC Simulation - Local Notebook Runner

Pairwise-CV protocol (same as the HPC masked scripts): 

outer doCrossValidationByPairwise on the OBSERVED matrix R_noisy, 

nested inner CV for HP selection, 

outer test scored against the clean truth R. 

Parallelised by opening several notebooks, each owning a slice of DATASET_IDS.

**Hightlight:**

- 2026.07.24: Added one specific parameterization strategy that is better for the algorithm results. 

## Cell 1 — Configure which datasets this notebook runs

In [12]:
# ============================================================
# USER SETTINGS — change these per notebook instance
# ============================================================

# A unique label for this notebook — used in the output filename
# e.g. 'nb0', 'nb1', 'nb2' ... open one notebook per label
NOTEBOOK_ID = "nb4"

# Which dataset_grid indices (i_ds) this notebook will process.
# dataset_grid has 100 entries (10 feature sizes x 10 repeats),
# indexed 0..99. Split them across notebooks however you like.
#
# Example splits across 4 notebooks:
#   nb0: list(range(0,  25))   # feature_id 0-1, all repeats
#   nb1: list(range(25, 50))   # feature_id 2-3, all repeats
#   nb2: list(range(50, 75))   # feature_id 4-6, all repeats
#   nb3: list(range(75, 100))  # feature_id 7-9, all repeats
#
# Or by repeat_id if you prefer:
#   nb0: [i for i in range(100) if (i % 10) < 5]   # repeat_id 0-4
#   nb1: [i for i in range(100) if (i % 10) >= 5]  # repeat_id 5-9

DATASET_IDS = list(range(40, 50))  # <-- change this per notebook

BASE_SEED = 123456  # must match HPC runs for reproducible splits

# ============================================================

## Cell 2 — Paths

In [13]:
import os
import sys
import gzip
import pickle
import warnings
import logging
from matplotlib import pyplot as plt

import numpy as np
from tqdm import TqdmSynchronisationWarning, tqdm

warnings.simplefilter("ignore", TqdmSynchronisationWarning)

# ====== user paths — adjust to your local machine ======
PATH_ROOT = "/Users/sijianfan/projects/BiSSGL"  # <-- change to local root
PATH_DATA = os.path.join(PATH_ROOT, "datasets/simulations/masked_ones")
PATH_OUTPUT = os.path.join(PATH_ROOT, "outputs/results/simulations/masked_ones")
PATH_ARCHIVE = os.path.join(PATH_OUTPUT, "archived")
DRIMC_PATH = os.path.join(PATH_ROOT, "scripts/methods/DRIMC")

for p in (PATH_OUTPUT, PATH_ARCHIVE):
    os.makedirs(p, exist_ok=True)

sys.path.append(PATH_ROOT)

print(f"Output will be saved to: {PATH_OUTPUT}")
print(f"Notebook ID: {NOTEBOOK_ID}  |  Datasets to run: {len(DATASET_IDS)}")

Output will be saved to: /Users/sijianfan/projects/BiSSGL/outputs/results/simulations/masked_ones
Notebook ID: nb4  |  Datasets to run: 10


## Cell 3 — Imports

In [14]:
from sgimc.utils import mc_split, get_submatrix
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import ParameterGrid, ShuffleSplit, train_test_split
from scipy.special import expit

## Cell 4 — R setup via rpy2

In [15]:
import rpy2.robjects as robjects

r = robjects.r
r(f'setwd("{DRIMC_PATH}")')

for pkg in ["matrixcalc", "data.table", "Rcpp", "ROCR", "Bolstad2", "MESS"]:
    r(f"library({pkg})")

for r_file in [
    "doCrossValidationByPairwise.R",
    "constrNeig.R",
    "calcLogLik.R",
    "calcDeriv.R",
    "updateUV.R",
]:
    r(f'source("{r_file}")')

r("library(Rcpp)")
for cpp_file in ["log1pexp.cpp", "sigmoid.cpp"]:
    r(f'sourceCpp("{cpp_file}")')

doCrossValidationByPairwise_R = r["doCrossValidationByPairwise"]
updateUV_R = r["updateUV"]
constrNeig_R = r["constrNeig"]
print("R setup complete.")

R setup complete.


## Cell 5 — Helper functions

In [16]:
def get_metrics(real_score, predict_score):
    sorted_predict_score = np.array(
        sorted(list(set(np.array(predict_score).flatten())))
    )
    sorted_predict_score_num = len(sorted_predict_score)
    thresholds = sorted_predict_score[
        np.int32(sorted_predict_score_num * np.arange(1, 1000) / 1000)
    ]
    thresholds = np.mat(thresholds)
    thresholds_num = thresholds.shape[1]
    predict_score_matrix = np.tile(predict_score, (thresholds_num, 1))
    negative_index = np.where(predict_score_matrix < thresholds.T)
    positive_index = np.where(predict_score_matrix >= thresholds.T)
    predict_score_matrix[negative_index] = 0
    predict_score_matrix[positive_index] = 1
    TP = predict_score_matrix.dot(real_score.T)
    FP = predict_score_matrix.sum(axis=1) - TP
    FN = real_score.sum() - TP
    TN = len(real_score.T) - TP - FP - FN
    fpr = FP / (FP + TN)
    tpr = TP / (TP + FN)
    ROC_dot_matrix = np.mat(sorted(np.column_stack((fpr, tpr)).tolist())).T
    ROC_dot_matrix.T[0] = [0, 0]
    ROC_dot_matrix = np.c_[ROC_dot_matrix, [1, 1]]
    x_ROC = ROC_dot_matrix[0].T
    y_ROC = ROC_dot_matrix[1].T
    auc = 0.5 * (x_ROC[1:] - x_ROC[:-1]).T * (y_ROC[:-1] + y_ROC[1:])
    recall_list = tpr
    precision_list = TP / (TP + FP)
    PR_dot_matrix = np.mat(
        sorted(np.column_stack((recall_list, precision_list)).tolist())
    ).T
    PR_dot_matrix.T[0] = [0, 1]
    PR_dot_matrix = np.c_[PR_dot_matrix, [1, 0]]
    x_PR = PR_dot_matrix[0].T
    y_PR = PR_dot_matrix[1].T
    aupr = 0.5 * (x_PR[1:] - x_PR[:-1]).T * (y_PR[:-1] + y_PR[1:])
    f1_score_list = 2 * TP / (len(real_score.T) + TP - TN)
    accuracy_list = (TP + TN) / len(real_score.T)
    specificity_list = TN / (TN + FP)
    max_index = np.argmax(f1_score_list)
    f1_score = f1_score_list[max_index]
    accuracy = accuracy_list[max_index]
    specificity = specificity_list[max_index]
    recall = recall_list[max_index]
    precision = precision_list[max_index]
    return [aupr[0, 0], auc[0, 0], f1_score, accuracy, recall, specificity, precision]

# ====== variable-selection accuracy vs the known truth ======
def selection_scores(est, n_true, tol=0.0):
    """Variable-selection metrics for a coefficient matrix `est` (d, K).

    A feature (row of `est`) is 'selected' if its row is not all-zero
    (max |.| > tol), matching the d1/d2 definition. The first `n_true`
    features are the ground-truth informative ones (the generator puts the
    signal in the first n_true columns of X and Y).

    Note: with many noise features, plain accuracy is dominated by true
    negatives -- recall, precision and f1 are the informative quantities.
    """
    d = est.shape[0]
    selected = np.abs(est).max(axis=1) > tol  # (d,) bool
    truth = np.zeros(d, dtype=bool)
    truth[:n_true] = True

    TP = int(np.sum(selected & truth))
    FP = int(np.sum(selected & ~truth))
    FN = int(np.sum(~selected & truth))
    TN = int(np.sum(~selected & ~truth))

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # = TP / n_true
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "accuracy": (TP + TN) / d,
        "precision": precision,
        "recall": recall,  # sensitivity / TPR over the true features
        "specificity": specificity,
        "f1": f1,
        "n_selected": int(np.sum(selected)),
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
    }

def to_r_matrix(arr):
    return r.matrix(
        robjects.FloatVector(arr.flatten()),
        byrow=True,
        nrow=arr.shape[0],
        ncol=arr.shape[1],
    )


def build_sim_matrices(U, V, K=3):
    simD = 1 - pairwise_distances(U[:, 0:24], metric="jaccard")
    simT = 1 - pairwise_distances(V[:, 0:24], metric="jaccard")
    simD = np.nan_to_num(simD, nan=0.0)
    simT = np.nan_to_num(simT, nan=0.0)
    np.fill_diagonal(simD, 1.0)
    np.fill_diagonal(simT, 1.0)
    lap = constrNeig_R(to_r_matrix(simD), to_r_matrix(simT), K=K)
    simD_R = lap.rx2("simD")
    simT_R = lap.rx2("simT")
    return np.array(simD_R), np.array(simT_R), simD_R, simT_R  # constrained np + R


def extract_fold(savedFolds, trial, fold):
    fold_data = savedFolds.rx2(trial + 1).rx2(fold + 1)
    Y_train = np.array(fold_data.rx2(7))
    test_label = np.array(fold_data.rx2(1)).flatten()
    test_row = np.array(fold_data.rx2(3)).flatten().astype(int) - 1
    test_col = np.array(fold_data.rx2(4)).flatten().astype(int) - 1
    known_drug_idx = np.array(fold_data.rx2(5)).flatten().astype(int)
    known_target_idx = np.array(fold_data.rx2(6)).flatten().astype(int)
    return Y_train, test_row, test_col, test_label, known_drug_idx, known_target_idx

_data_cache = {}  # i_ds -> (U, V, R_true, R_obs)
_outer_folds_cache = {}  # i_ds -> savedFolds (R object) for R_obs
_inner_folds_cache = {}  # (i_ds, fold) -> inner savedFolds (R object)


print("Helpers defined.")

Helpers defined.


## Cell 6 — Build grids

In [17]:
n_features_grid = np.arange(50, 501, 50)
n_repeats = 10
filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

dataset_grid = []
for feature_id, n_features in enumerate(n_features_grid):
    for repeat_id in range(n_repeats):
        dataset_grid.append(
            {
                "feature_id": int(feature_id),
                "n_features": int(n_features),
                "repeat_id": int(repeat_id),
                "filename": os.path.join(
                    PATH_DATA, filename_template.format(n_features, repeat_id)
                ),
            }
        )

# CV config
KFOLD = 10
NUMSPLIT_OUTER = 1
N_INNER_KFOLD = 10
N_INNER_EVAL = 2
DO_INNER_CV = True
N_RANK = 25  # number of informative (true) features: first N_RANK rows of est_A/est_B

# ====== model hyperparameter grid (simulation grid) ======
grid_model = ParameterGrid(
    {
        "tilde_lambda0": [1000],
        "tilde_lambda1": [100],
        # "lambda0": [10, 50, 100, 500, 1000], --- IGNORE ---
        "lambda1": [100],
        "xi": [10, 100, 1000],
        "eta": [1e-5],
        "K": [25],
    }
)

n_m = len(list(grid_model))

my_ids = list(DATASET_IDS)
print(f"Datasets assigned : {len(my_ids)}")
print(f"Folds per dataset : {KFOLD}")
print(f"Hyperparam combos : {n_m}")
print(f"Total fits (outer): {len(my_ids) * KFOLD * n_m}")

Datasets assigned : 10
Folds per dataset : 10
Hyperparam combos : 3
Total fits (outer): 300


## Cell 7 — Run

Progress bars are shown per dataset.  
Results are **incrementally saved** after each dataset finishes — if the kernel dies mid-run you keep completed datasets.

In [18]:
from scripts.methods.BiSSGL.BiSSGLc import BiSSGL

In [19]:
outfile = os.path.join(PATH_OUTPUT, f"results_bissgl_local_{NOTEBOOK_ID}.gz")
all_results = []
 
for i_ds in tqdm(my_ids, desc="Datasets"):
    ds = dataset_grid[i_ds]
 
    with gzip.open(ds["filename"], "rb") as fin:
        data = pickle.load(fin)
    U = np.asarray(data["X"], dtype=float)
    V = np.asarray(data["Y"], dtype=float)
    R_true = np.asarray(data["R"], dtype=float)        # clean truth {0,1}
    R_obs = np.asarray(data["R_noisy"], dtype=float)   # observed {0,1}
 
    # outer pairwise folds on the OBSERVED matrix — once per dataset
    # (same seed formula as the DRIMC/NRLMF/HPC runners -> identical folds)
    outer_seed = (BASE_SEED + i_ds * 7919) % (2**31 - 1)
    savedFolds = doCrossValidationByPairwise_R(
        to_r_matrix(R_obs), kfold=KFOLD, numSplit=NUMSPLIT_OUTER,
        seeds=robjects.IntVector([outer_seed]),
    )
 
    _inner_folds_cache = {}
    dataset_results = []
 
    for fold in range(KFOLD):
        Y_train_dense, test_row, test_col, _obs_label, _, _ = extract_fold(
            savedFolds, 0, fold
        )
        test_label_true = R_true[test_row, test_col]
 
        for i_m, par_mdl in enumerate(grid_model):
            combo_idx = i_ds * KFOLD * n_m + fold * n_m + i_m
 
            # extract model params
            xi = par_mdl["xi"]
            eta = par_mdl["eta"]
            tilde_lambda0 = par_mdl["tilde_lambda0"]
            tilde_lambda1 = par_mdl["tilde_lambda1"]
            lambda0 = par_mdl["tilde_lambda0"]   # mirrors simulation behaviour
            lambda1 = par_mdl["lambda1"]
            K = par_mdl["K"]
 
            try:
                # ---- outer fit -> test score (vs CLEAN truth R) ----
                # BiSSGL uses {0,1} encoding -- foldMat is already 0/1
                model = BiSSGL(
                    Y=Y_train_dense, U=U, V=V,
                    xi=xi, eta=eta,
                    tilde_lambda0=tilde_lambda0, tilde_lambda1=tilde_lambda1,
                    tilde_alpha=1 / K, tilde_beta=1,
                    lambda0=lambda0, lambda1=lambda1,
                    alpha=1 / K, beta=1, K=K,
                    max_iter=1000, tol=1e-5,
                )
                _, est_A, est_B, logLik = model.optimization()
 
                UA = U @ est_A
                VB = V @ est_B
                prob_full = expit(UA @ VB.T)
                prob_test = prob_full[test_row, test_col]
                scores_test = get_metrics(
                    np.asarray(test_label_true, dtype=float),
                    np.asarray(prob_test, dtype=float),
                )
                d1_test = int(sum(abs(est_A).max(axis=1) > 0))
                d2_test = int(sum(abs(est_B).max(axis=1) > 0))
                sel_A = np.where(abs(est_A).max(axis=1) > 0)[0].astype(int).tolist()
                sel_B = np.where(abs(est_B).max(axis=1) > 0)[0].astype(int).tolist()
                # variable-selection accuracy vs the known first-N_RANK truth
                sel_acc_A = selection_scores(est_A, N_RANK)
                sel_acc_B = selection_scores(est_B, N_RANK)
 
                # ---- inner CV for HP selection (OBSERVED labels) ----
                inner_scores = []
                if DO_INNER_CV:
                    if fold not in _inner_folds_cache:
                        inner_seed = (BASE_SEED + i_ds * 7919 + fold + 31415) % (2**31 - 1)
                        _inner_folds_cache[fold] = doCrossValidationByPairwise_R(
                            to_r_matrix(Y_train_dense),
                            kfold=N_INNER_KFOLD, numSplit=1,
                            seeds=robjects.IntVector([inner_seed]),
                        )
                    inner_savedFolds = _inner_folds_cache[fold]
 
                    for cv in range(N_INNER_EVAL):
                        Y_inner, val_row, val_col, val_label, _, _ = extract_fold(
                            inner_savedFolds, 0, cv
                        )
                        model_cv = BiSSGL(
                            Y=Y_inner, U=U, V=V,
                            xi=xi, eta=eta,
                            tilde_lambda0=tilde_lambda0, tilde_lambda1=tilde_lambda1,
                            tilde_alpha=1 / K, tilde_beta=1,
                            lambda0=lambda0, lambda1=lambda1,
                            alpha=1 / K, beta=1, K=K,
                            max_iter=1000, tol=1e-5,
                        )
                        _, est_A_cv, est_B_cv, logLik_cv = model_cv.optimization()
 
                        UA_cv = U @ est_A_cv
                        VB_cv = V @ est_B_cv
                        prob_full_cv = expit(UA_cv @ VB_cv.T)
                        prob_valid = prob_full_cv[val_row, val_col]
                        scores_valid = get_metrics(
                            np.asarray(val_label, dtype=float),
                            np.asarray(prob_valid, dtype=float),
                        )
                        inner_scores.append((int(cv), scores_valid,
                                             int(sum(abs(est_A_cv).max(axis=1) > 0)),
                                             int(sum(abs(est_B_cv).max(axis=1) > 0))))
                else:
                    inner_scores.append((-1, [np.nan] * 7, -1, -1))
 
                for cv, scores_valid, d1_valid, d2_valid in inner_scores:
                    dataset_results.append(
                        {
                            "feature_id": ds["feature_id"],
                            "n_features": ds["n_features"],
                            "repeat_id": ds["repeat_id"],
                            "fold": int(fold),
                            "xi": xi, "eta": eta,
                            "tilde_lambda0": tilde_lambda0,
                            "tilde_lambda1": tilde_lambda1,
                            "lambda0": lambda0, "lambda1": lambda1,
                            "K": K,
                            "cv": int(cv),
                            "val_score": scores_valid,
                            "val_d1": d1_valid, "val_d2": d2_valid,
                            "test_score": scores_test,
                            "test_d1": d1_test, "test_d2": d2_test,
                            "test_sel_A": sel_A, "test_sel_B": sel_B,
                            "test_selacc_A": sel_acc_A,
                            "test_selacc_B": sel_acc_B,
                        }
                    )
 
            except Exception as e:
                print(f"  ERROR i_ds={i_ds} fold={fold} "
                      f"xi={xi} tilde_lambda0={tilde_lambda0}: {e}")
 
    # ---- incremental save after each dataset ----
    all_results.extend(dataset_results)
    with gzip.open(outfile, "wb+", 4) as fout:
        pickle.dump(all_results, fout)
    print(f"  i_ds={i_ds} done — {len(all_results)} rows so far -> {outfile}")
 
print(f"\nDone. Total rows: {len(all_results)}")

Datasets:   0%|          | 0/10 [00:00<?, ?it/s]

Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished

Datasets:  10%|█         | 1/10 [1:47:47<16:10:04, 6467.11s/it]

  i_ds=40 done — 60 rows so far -> /Users/sijianfan/projects/BiSSGL/outputs/results/simulations/masked_ones/results_bissgl_local_nb4.gz
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999


Datasets:  20%|██        | 2/10 [3:35:12<14:20:35, 6454.48s/it]

...

  i_ds=47 done — 480 rows so far -> /Users/sijianfan/projects/BiSSGL/outputs/results/simulations/masked_ones/results_bissgl_local_nb4.gz
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999

Datasets:  90%|█████████ | 9/10 [15:07:46<1:31:45, 5505.60s/it]

  i_ds=48 done — 540 rows so far -> /Users/sijianfan/projects/BiSSGL/outputs/results/simulations/masked_ones/results_bissgl_local_nb4.gz
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999
Finished with iterations of 999

Datasets: 100%|██████████| 10/10 [16:19:01<00:00, 5874.15s/it] 

  i_ds=49 done — 600 rows so far -> /Users/sijianfan/projects/BiSSGL/outputs/results/simulations/masked_ones/results_bissgl_local_nb4.gz

Done. Total rows: 600
